## SQL - JOINING DATA

| Join Type                    | What it Returns                          | Matching Required? | Non-Matching Rows Included? | Use Case                       |
| ---------------------------- | ---------------------------------------- | ------------------ | --------------------------- | ------------------------------ |
| **INNER JOIN**               | Only matching rows from both tables      | ✅ Yes              | ❌ No                        | When you need only common data |
| **LEFT JOIN (LEFT OUTER)**   | All rows from left + matching from right | ❌ No               | ✅ Left table rows kept      | When left table is primary     |
| **RIGHT JOIN (RIGHT OUTER)** | All rows from right + matching from left | ❌ No               | ✅ Right table rows kept     | When right table is primary    |
| **FULL JOIN (FULL OUTER)**   | All rows from both tables                | ❌ No               | ✅ Both sides                | When you need everything       |
| **CROSS JOIN**               | Cartesian product (all combinations)     | ❌ No               | ✅ All combinations          | Rare, combinations             |
| **SELF JOIN**                | Table joined with itself                 | Depends            | Depends                     | Hierarchical data              |


In [1]:
import pandas as pd
import seaborn as sb
import sqlite3

In [15]:
# Create database
conn = sqlite3.connect('joins_practice.db')
cursor=conn.cursor()
conn.commit()

In [12]:
# Create sample tables
students = pd.DataFrame({
    'student_id': [1, 2, 3, 4, 5],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'class_id': [101, 102, 101, None, 103]
})

classes = pd.DataFrame({
    'class_id': [101, 102, 104],
    'class_name': ['Math', 'Science', 'History'],
    'teacher': ['Mr. Smith', 'Ms. Johnson', 'Mr. Brown']
})

grades = pd.DataFrame({
    'student_id': [1, 2, 3, 6],
    'subject': ['Math', 'Science', 'Math', 'English'],
    'grade': ['A', 'B', 'A', 'C']
})

# Save to database
students.to_sql('students', conn, if_exists='replace', index=False)
classes.to_sql('classes', conn, if_exists='replace', index=False)
grades.to_sql('grades', conn, if_exists='replace', index=False)

print("Sample tables created!")
print(f"Students: {len(students)} rows")
print(f"Classes: {len(classes)} rows")
print(f"Grades: {len(grades)} rows")


Sample tables created!
Students: 5 rows
Classes: 3 rows
Grades: 4 rows


In [ ]:
conn.close()

In [21]:
## Inner join in SQL via Python Pandas
query="SELECT a.STUDENT_ID,a.NAME,b.subject,b.grade from students a inner join grades b on a.student_id=b.student_id"

df = pd.read_sql(query,conn)
print(df)

## Inner join in Python Pandas
##Always include the joining columns in both the tables, because it's like creating a new dataframe and applying 
#join on the new frames.
#right_on is the join colummn that we mention on right side
#left_on is the join colummn that we mention on left side

pd.merge(students[['student_id','name']]
         ,grades[['student_id','subject','grade']]
         ,right_on="student_id"
         ,left_on="student_id"
         ,how="inner")



   student_id     name  subject grade
0           1    Alice     Math     A
1           2      Bob  Science     B
2           3  Charlie     Math     A


,student_id,name,subject,grade
0,1,Alice,Math,A
1,2,Bob,Science,B
2,3,Charlie,Math,A


In [22]:
## Left join in SQL via Python Pandas
query="SELECT a.STUDENT_ID,a.NAME,b.subject,b.grade from students a left join grades b on a.student_id=b.student_id"

df = pd.read_sql(query,conn)
print(df)

## Inner join in Python Pandas
##Always include the joining columns in both the tables, because it's like creating a new dataframe and applying 
#join on the new frames.
#right_on is the join colummn that we mention on right side
#left_on is the join colummn that we mention on left side

pd.merge(students[['student_id','name']]
         ,grades[['student_id','subject','grade']]
         ,right_on="student_id"
         ,left_on="student_id"
         ,how="left")



   student_id     name  subject grade
0           1    Alice     Math     A
1           2      Bob  Science     B
2           3  Charlie     Math     A
3           4    Diana      NaN   NaN
4           5      Eve      NaN   NaN


,student_id,name,subject,grade
0,1,Alice,Math,A
1,2,Bob,Science,B
2,3,Charlie,Math,A
3,4,Diana,NaN,NaN
4,5,Eve,NaN,NaN


In [23]:
## Right join in SQL via Python Pandas
query="SELECT a.STUDENT_ID,a.NAME,b.subject,b.grade from students a right join grades b on a.student_id=b.student_id"

df = pd.read_sql(query,conn)
print(df)

## Inner join in Python Pandas
##Always include the joining columns in both the tables, because it's like creating a new dataframe and applying 
#join on the new frames.
#right_on is the join colummn that we mention on right side
#left_on is the join colummn that we mention on left side

pd.merge(students[['student_id','name']]
         ,grades[['student_id','subject','grade']]
         ,right_on="student_id"
         ,left_on="student_id"
         ,how="right")

   student_id     name  subject grade
0         1.0    Alice     Math     A
1         2.0      Bob  Science     B
2         3.0  Charlie     Math     A
3         NaN      NaN  English     C


,student_id,name,subject,grade
0,1,Alice,Math,A
1,2,Bob,Science,B
2,3,Charlie,Math,A
3,6,NaN,English,C


In [30]:
##-------------------MULTIPLE JOIN IN SQL AND PANDAS-----------------------------
query="SELECT a.STUDENT_ID,a.NAME,c.class_id,c.class_name,c.teacher from students a INNER join grades \
b on a.student_id=b.student_id inner join classes c \
on a.class_id=c.class_id"

df = pd.read_sql(query,conn)
print(df)

join1 = pd.merge(students[['student_id','name','class_id']]
         ,grades[['student_id','subject','grade']]
         ,right_on="student_id"
         ,left_on="student_id"
         ,how="inner")

join2 = pd.merge(join1[['student_id','name','class_id']]
         ,classes[['class_id','class_name','teacher']]
         ,right_on="class_id"
         ,left_on="class_id"
         ,how="inner")
join2

   student_id     name  class_id class_name      teacher
0           1    Alice       101       Math    Mr. Smith
1           2      Bob       102    Science  Ms. Johnson
2           3  Charlie       101       Math    Mr. Smith


,student_id,name,class_id,class_name,teacher
0,1,Alice,101.0,Math,Mr. Smith
1,2,Bob,102.0,Science,Ms. Johnson
2,3,Charlie,101.0,Math,Mr. Smith


In [10]:
conn = sqlite3.connect('joins_practice.db')
#Load Titanic dataset
titanic = sb.load_dataset('titanic')
#Create passengers table (main info)
passengers = titanic[['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']].copy()
passengers ['passenger_id'] = range(1, len (passengers) + 1)
passengers.to_sql('passengers', conn, if_exists='replace', index=False)
#Create embarkation_ports table (lookup table)
ports = pd.DataFrame({
'port_code': ['S', 'C', 'Q'],
'port_name': ['Southampton', 'Cherbourg', 'Queenstown'],
'country': ['England', 'France', 'Ireland']
})
ports.to_sql('embarkation_ports', conn, if_exists='replace', index=False)

# Create ticket_classes table (lookup table)
ticket_classes = pd. DataFrame({
'pclass': [1, 2, 3],
'class_name': ['First', 'Second', 'Third'],
'description': ['Upper class', 'Middle class', 'Lower class']
})
ticket_classes.to_sql('ticket_classes', conn, if_exists='replace', index=False)
print("\n Titanic tables created!")
print(f"Passengers: {(len(passengers))} rows")
print(f"Embarkation Ports: {len(ports)} rows")
print(f"Ticket Classes: {(len(ticket_classes))} rows")
conn.close()


 Titanic tables created!
Passengers: 891 rows
Embarkation Ports: 3 rows
Ticket Classes: 3 rows
